
# 23 — Preference Data, Sequence Log-Probabilities, and Reward Modeling

## Goal

Understand how language-model post-training can learn from
preferences between complete responses.

We will build preference data representations, calculate sequence
log-probabilities, derive the Bradley-Terry preference objective,
and train a small reward model.

## 01. Preference Data and Pairwise Comparisons

A preference example contains a prompt, a chosen response,
and a rejected response.

Preference is conditional on the prompt. It does not imply
that one response is universally better than another.


In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PreferenceExample:
    """Represent one prompt-conditioned pairwise preference.

    Args:
        prompt: Instruction shared by both responses.
        chosen: Preferred response.
        rejected: Less preferred response.
    """

    prompt: str
    chosen: str
    rejected: str


preferences = [
    PreferenceExample("Say yes", "Yes", "No"),
    PreferenceExample("Say no", "No", "Yes"),
    PreferenceExample("Say hello", "Hello", "Farewell"),
]
for example in preferences:
    print(example)

PreferenceExample(prompt='Say yes', chosen='Yes', rejected='No')
PreferenceExample(prompt='Say no', chosen='No', rejected='Yes')
PreferenceExample(prompt='Say hello', chosen='Hello', rejected='Farewell')


In [5]:
# Validate basic pairwise preference structure.
for example in preferences:
    assert example.prompt
    assert example.chosen
    assert example.rejected
    assert example.chosen != example.rejected

# The same response ranking can reverse under another prompt.
assert preferences[0].chosen == preferences[1].rejected
assert preferences[0].rejected == preferences[1].chosen
assert preferences[0].prompt != preferences[1].prompt

print("Prompt-conditioned preference test passed.")


Prompt-conditioned preference test passed.



### Preference Data — Takeaways

Supervised fine-tuning learns from selected reference responses.

Preference learning uses relative judgments between candidate
responses conditioned on a shared prompt.

A preference pair does not directly provide an absolute reward,
nor does it specify the probability of either response.

Response ranking may reverse when the prompt changes.

The next step is to calculate token-level and complete-response
log-probabilities under a language model.



## 02. Token vs Sequence Log-Probabilities

### Why complete-response probabilities?

A causal language model predicts one token at a time.

For a prompt x and completion y, the probability chain rule gives:

$$
\log \pi_\theta(y \mid x)
=
\sum_{t=0}^{T_y-1}
\log \pi_\theta(y_t \mid x,y_{<t}).
$$

Sequence log-probability is a sum, not a mean.

The following pipeline converts GPT logits into a score for a
complete candidate response.

```mermaid
flowchart TD
    A["Prompt x + Completion y"] --> B["Tokenize full sequence"]
    B --> C["Shift inputs and labels once"]
    C --> D["GPT forward: logits (B, T, V)"]
    D --> E["log_softmax over vocabulary V"]
    E --> F["Gather log p of actual target token"]
    F --> G["Mask prompt and padding positions"]
    G --> H["Sum completion-token log probabilities"]
    H --> I["log π(y | x)"]
```

This is teacher-forced scoring. The model receives the actual
previous response tokens when scoring each target character.

The resulting sequence log-probability is a language-model
likelihood, not a learned preference reward.


In [6]:
import torch

# (T=2,): conditional probabilities of two successive target tokens
token_probabilities = torch.tensor([0.8, 0.9])

# (): product of the conditional token probabilities
sequence_probability = token_probabilities.prod()

# (): sum of the conditional token log-probabilities
sequence_log_probability = token_probabilities.log().sum()

torch.testing.assert_close(sequence_log_probability, sequence_probability.log())

print("Sequence probability:", sequence_probability.item())
print("Sequence log-probability:", sequence_log_probability.item())


Sequence probability: 0.7199999690055847
Sequence log-probability: -0.3285040855407715


In [7]:
from pathlib import Path

from llmfp.data import (
    CharacterTokenizer,
    pad_supervised_batch,
    shift_supervised_tokens,
)
from llmfp.models import GPT, GPTConfig
from llmfp.training import per_example_sft_nll

checkpoint_path = Path("../checkpoints/tiny_shakespeare_sft/step_00040.pt")

if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"Missing {checkpoint_path}. "
        "Complete the checkpoint-saving cell in Lesson 22 first."
    )

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

tokenizer = CharacterTokenizer(checkpoint["tokenizer_vocabulary"])

config = GPTConfig(**checkpoint["model_config"])

sft_policy = GPT(config)

sft_policy.load_state_dict(checkpoint["model_state_dict"], strict=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sft_policy = sft_policy.to(device)
sft_policy.eval()

assert tokenizer.vocab_size == config.vocab_size == 65

print("Loaded SFT checkpoint:", checkpoint_path.name)
print("Vocabulary size:", config.vocab_size)
print("Device:", device)


Loaded SFT checkpoint: step_00040.pt
Vocabulary size: 65
Device: cuda


In [8]:
candidate_examples = []
candidate_metadata = []

for pair_index, example in enumerate(preferences):
    prompt_text = f"User: {example.prompt}\nAssistant: "

    for kind, response in (
        ("chosen", example.chosen),
        ("rejected", example.rejected),
    ):
        completion_text = response + "\n"
        full_text = prompt_text + completion_text

        # (L,): serialized prompt and completion character IDs
        full_ids = torch.tensor(tokenizer.encode(full_text), dtype=torch.long)

        # (L,): full-sequence positions selected for supervision
        target_mask = torch.zeros_like(full_ids, dtype=torch.bool)

        prompt_length = len(tokenizer.encode(prompt_text))
        target_mask[prompt_length:] = True

        # (T=L-1,): causally aligned inputs and completion-only labels
        input_ids, labels = shift_supervised_tokens(full_ids, target_mask)

        # (R,): supervised response character IDs, including newline
        selected_labels = labels[labels.ne(-100)]

        torch.testing.assert_close(
            selected_labels,
            torch.tensor(tokenizer.encode(completion_text), dtype=torch.long),
        )

        candidate_examples.append((input_ids, labels))
        candidate_metadata.append(
            {
                "pair_index": pair_index,
                "kind": kind,
                "prompt": example.prompt,
                "response": response,
                "completion": completion_text,
            }
        )

# Use a valid in-vocabulary character for right padding.
pad_token_id = tokenizer.token_to_id[" "]

# (B=6, T_max): padded predictor inputs
batch_inputs, batch_labels = pad_supervised_batch(
    candidate_examples, pad_token_id=pad_token_id
)

assert batch_inputs.shape == batch_labels.shape
assert batch_inputs.shape[0] == 6
assert batch_labels.ne(-100).sum().item() == 29

print("Candidate batch:", batch_inputs.shape)
print("Total supervised characters:", 29)


Candidate batch: torch.Size([6, 35])
Total supervised characters: 29


In [9]:
import torch.nn.functional as F

# (B=6, T_max): predictor inputs on the model device
inputs_device = batch_inputs.to(device)

# (B=6, T_max): supervised labels on the model device
labels_device = batch_labels.to(device)

with torch.no_grad():
    # (B=6, T_max, V=65): GPT next-token predictions
    logits = sft_policy(inputs_device)

    # (B=6, T_max, V=65): normalized vocabulary log-probabilities
    log_probs = F.log_softmax(logits, dim=-1)

    # (B=6, T_max): supervised completion positions
    completion_mask = labels_device.ne(-100)

    # (B=6, T_max): safe indices for gathering vocabulary values
    safe_labels = labels_device.masked_fill(~completion_mask, 0)

    # (B=6, T_max): log-probability of the actual next token
    selected_log_probs = log_probs.gather(
        dim=-1, index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)

    # (B=6, T_max): zero out prompt and padding contributions
    token_log_probs = selected_log_probs.masked_fill(~completion_mask, 0.0)

    # (B=6,): log-probability of each full completion
    sequence_log_probs = token_log_probs.sum(dim=1)

    # (B=6,): supervised token counts per completion
    completion_lengths = completion_mask.sum(dim=1)

    # (B=6,): mean NLL, using our Lesson 22 reusable primitive
    mean_nll = per_example_sft_nll(logits, labels_device)

# The sequence score must equal minus mean NLL times token count.
torch.testing.assert_close(sequence_log_probs, -mean_nll * completion_lengths)

assert torch.isfinite(sequence_log_probs).all()

print("Sequence log-probabilities:")
for metadata, score in zip(
    candidate_metadata,
    sequence_log_probs.cpu().tolist(),
):
    print(
        f"{metadata['prompt']:12s} "
        f"{metadata['kind']:8s} "
        f"{metadata['response']:10s} "
        f"logp={score:.4f}"
    )

print("Sequence log-probability consistency passed.")


Sequence log-probabilities:
Say yes      chosen   Yes        logp=-0.0007
Say yes      rejected No         logp=-20.4530
Say no       chosen   No         logp=-18.6250
Say no       rejected Yes        logp=-0.0015
Say hello    chosen   Hello      logp=-0.0011
Say hello    rejected Farewell   logp=-10.3109
Sequence log-probability consistency passed.


In [10]:
import plotly.graph_objects as go

fig = go.Figure()

for row, kind in ((0, "chosen"), (1, "rejected")):
    metadata = candidate_metadata[row]

    # (R,): log-probabilities of actual completion characters
    values = token_log_probs[row, completion_mask[row]].cpu().tolist()

    characters = list(metadata["completion"])

    assert len(values) == len(characters)

    fig.add_trace(
        go.Bar(
            x=list(range(1, len(values) + 1)),
            y=values,
            name=f"{kind.title()}: {metadata['response']}",
            customdata=[repr(char) for char in characters],
            hovertemplate=(
                "Character position: %{x}"
                "<br>Character: %{customdata}"
                "<br>Log-probability: %{y:.4f}"
                "<extra></extra>"
            ),
        )
    )

fig.update_layout(
    title="Token Log-Probabilities — Prompt: Say yes",
    xaxis_title="Completion character position",
    yaxis_title="Log-probability of target character",
    xaxis=dict(dtick=1),
    barmode="group",
    height=420,
)

fig.show()



### Token and Sequence Log-Probabilities — Takeaways

The probability of a completion is the product of its conditional
next-token probabilities.

Taking the logarithm converts that product into a sum.

We restored the SFT GPT from Lesson 22 and teacher-forced both
chosen and rejected responses through the same policy.

`F.log_softmax` converted vocabulary logits into log-probabilities.

`gather` selected the log-probability of each actual target token.
Ignored labels were replaced with valid gather indices and then
masked out of the result.

Summing the completion-token log-probabilities produced one
sequence log-probability per candidate response.

We verified that sequence log-probability equals negative mean
NLL multiplied by the number of supervised completion tokens.

A policy's sequence log-probability is not a preference reward.
The model is not guaranteed to prefer the annotated chosen response.

The next section examines completion masks, sequence lengths,
and the difference between sum and mean normalization.
